In [1]:
import pandas as pd
import numpy as np

In [2]:
app_train = pd.read_csv('../data/app_train_eda.csv')
app_test = pd.read_csv('../data/app_test_eda.csv')

## Drop the outliers

In [3]:
upper = app_train['AMT_INCOME_TOTAL'].quantile(0.99)
app_train['AMT_INCOME_TOTAL'] = app_train['AMT_INCOME_TOTAL'].clip(upper=upper)

app_train = app_train[app_train['DAYS_EMPLOYED'] != app_train['DAYS_EMPLOYED'].max()]

## Deal with NaNs

In [4]:
# Если у человека нет машины, то, очевидно, у него нет и возраста автомобиля, поэтому просто заменим NaN на 0
app_train['OWN_CAR_AGE'] = app_train['OWN_CAR_AGE'].fillna(0)
app_test['OWN_CAR_AGE'] = app_test['OWN_CAR_AGE'].fillna(0)

In [5]:
# Сразу же удалим столбцы, в которых слишком большой процент пропусков
missing_pct = (app_train.isnull().sum() / len(app_train) * 100).sort_values(ascending=False)
missing_pct = missing_pct[missing_pct > 1]

In [6]:
# Пока что выберем 70 процентов, опираясь на график из EDA
cols_to_drop = missing_pct.index[missing_pct > 70].tolist()
app_train = app_train.drop(columns=cols_to_drop)
app_test = app_test.drop(columns=cols_to_drop)

if 'TARGET' in app_train.columns:
    target = app_train['TARGET']
    app_train.drop(columns=['TARGET'], inplace=True)

## Polinomial features

Выберем только важные признаки и лишь из них сделаем полиномы, во избежание переобучения:

In [ ]:
# Cоздадим новый датафрейм для полиномиальных признаков
poly_features = app_train[['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3', 'AGE_YEARS']]
poly_features_test = app_test[['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3', 'AGE_YEARS']]

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

# Создадим полиномиальный объект степени 3
poly_transformer = PolynomialFeatures(degree = 3)

# Тренировка полиномиальных признаков
poly_transformer.fit(poly_features)

# Трансформация признаков
poly_features = poly_transformer.transform(poly_features)
poly_features_test = poly_transformer.transform(poly_features_test)
print('Формат полиномиальных признаков: ', poly_features.shape)

Присвоим признакам имена при помощи метода get_feature_names:


In [ ]:
poly_transformer.get_feature_names_out(input_features = ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3', 'AGE_YEARS'])[:15]

Итого 35 полиномиальных и производных признаков. Проверих их корреляцию с таргетом.

In [ ]:
# Датафрейм для новых фич
poly_features = pd.DataFrame(poly_features,
                             columns = poly_transformer.get_feature_names_out(['EXT_SOURCE_1', 'EXT_SOURCE_2',
                                                                           'EXT_SOURCE_3', 'AGE_YEARS']))
poly_features_test = pd.DataFrame(poly_features_test,
                                  columns = poly_transformer.get_feature_names_out(['EXT_SOURCE_1', 'EXT_SOURCE_2',
                                                                                'EXT_SOURCE_3', 'AGE_YEARS']))

# Добавим таргет
poly_features_with_TARGET = poly_features.copy()
poly_features_with_TARGET['TARGET'] = target

# Рассчитаем корреляцию
poly_corrs = poly_features_with_TARGET.corr()['TARGET'].sort_values()

# Отобразим признаки с наивысшей корреляцией
print(poly_corrs.head(10))
print(poly_corrs.tail(5))

Итак, некоторые признаки показывают более высокую корреляцию, чем исходные. Есть смысл попробовать обучение с ними и без них. Для этого присоединим полиномиальные признаки к основному DF.

In [ ]:
# Объединим тренировочные датафреймы
poly_features['SK_ID_CURR'] = app_train['SK_ID_CURR']
app_train_poly = app_train.merge(poly_features, on = 'SK_ID_CURR', how = 'left')

# Объединим тестовые датафреймы
poly_features_test['SK_ID_CURR'] = app_test['SK_ID_CURR']
app_test_poly = app_test.merge(poly_features_test, on = 'SK_ID_CURR', how = 'left')

# Посмотрим формат
print('Тренировочная выборка с полиномиальными признаками: ', app_train_poly.shape)
print('Тестовая выборка с полиномиальными признаками: ', app_test_poly.shape)